In [0]:
# =============================================================================
# CELL #1: IMPORTS AND CONFIGURATION
# PURPOSE: Setup for historical data chunk automation for YFinance Data
# =============================================================================

from datetime import datetime, timedelta, date
import time

# Install required packages
%pip install yfinance

# Data processing imports
import yfinance as yf
from pyspark.sql import functions as F, Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

# Portfolio configuration (same as Bronze layer)
portfolio_config = {
    "tier_1": ["NVDA", "MSFT", "GOOGL", "AMZN", "META", "AAPL"],
    "tier_2": ["AMD", "CRM", "ORCL"], 
    "tier_3": ["PLTR", "AI", "SNOW", "MDB", "SMCI"],
    "benchmark": ["BOTZ"]
}

# Flatten all symbols
all_symbols = []
for tier_symbols in portfolio_config.values():
    all_symbols.extend(tier_symbols)

print(f"🏗️ Historical Chunk Automation Initialized")
print(f"📊 Portfolio: {len(all_symbols)} stocks")
print(f"🎯 Target: Complete historical data (1972-2024)")

In [0]:
# =============================================================================
# CELL #2: SMART HISTORICAL GROUPS
# PURPOSE: Group stocks by IPO era for efficient historical data collection
# =============================================================================

historical_groups = [
    {
        "group": 1,
        "name": "The Ancients (AMD)",
        "stocks": ["AMD"],
        "start": "1972-09-27",
        "end": "2024-08-01",
        "description": "53 years of semiconductor history"
    },
    {
        "group": 2, 
        "name": "1980s Tech Giants",
        "stocks": ["AAPL", "ORCL", "MSFT"],
        "start": "1980-12-12",
        "end": "2024-08-01", 
        "description": "45 years of personal computing revolution"
    },
    {
        "group": 3,
        "name": "Internet & GPU Era", 
        "stocks": ["AMZN", "NVDA", "CRM", "GOOGL"],
        "start": "1997-05-15",
        "end": "2024-08-01",
        "description": "28 years of internet and graphics revolution"
    },
    {
        "group": 4,
        "name": "Social & Cloud Era",
        "stocks": ["SMCI", "META", "BOTZ", "MDB"],
        "start": "2007-03-29", 
        "end": "2024-08-01",
        "description": "18 years of social media and cloud computing"
    },
    {
        "group": 5,
        "name": "AI Revolution",
        "stocks": ["SNOW", "PLTR", "AI"],
        "start": "2020-09-16",
        "end": "2024-08-01",
        "description": "5 years of AI and data analytics boom"
    }
]

for group in historical_groups:
    print(f"📊 Group {group['group']}: {group['name']}")
    print(f"   Stocks: {group['stocks']}")
    print(f"   Period: {group['start']} to {group['end']}")
    print()

In [0]:
# =============================================================================
# CELL #3: BRONZE SCHEMA AND GROUP FUNCTIONS
# =============================================================================

import pandas as pd

bronze_schema = StructType([
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("symbol", StringType(), True),
    StructField("date", StringType(), True),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", LongType(), True),
    StructField("data_source", StringType(), True)
])

def download_group_data(symbols, start_date, end_date, group_info):
    """Download historical data for a specific group period"""
    
    print(f"\n📥 Starting group download:")
    print(f"Group: {group_info['group']} ({group_info['name']})")
    print(f"Period: {start_date} to {end_date}")
    print(f"Symbols: {symbols} ({len(symbols)} stocks)")
    
    try:
        group_data = yf.download(
            symbols,
            start=start_date,
            end=end_date,
            group_by='ticker',
            progress=True
        )
        
        if group_data.empty:
            print(f"⚠️ No data returned for group {group_info['group']}")
            return None
            
        print(f"✅ Group download completed")
        print(f"Data shape: {group_data.shape}")
        return group_data
        
    except Exception as e:
        print(f"❌ Group download failed: {e}")
        return None

def process_group_to_bronze(group_data, symbols, group_info):
    """Convert group data to Bronze format"""
    
    print(f"🔄 Processing group {group_info['group']} to Bronze format...")
    print(f"📊 Data columns: {list(group_data.columns)}")
    
    all_bronze_rows = []
    ingestion_time = datetime.now()
    
    # Process each symbol
    for symbol in symbols:
        print(f"  Processing {symbol}...")
        
        # For multiple stocks, access like: group_data[symbol]['Open']
        symbol_data = group_data[symbol]
        
        # Convert each row to Bronze format
        for date_idx, row in symbol_data.iterrows():
            bronze_row = Row(
                ingestion_timestamp=ingestion_time,
                symbol=symbol,
                date=str(date_idx.date()),
                open=float(row['Open']) if not pd.isna(row['Open']) else None,
                high=float(row['High']) if not pd.isna(row['High']) else None,
                low=float(row['Low']) if not pd.isna(row['Low']) else None,
                close=float(row['Close']) if not pd.isna(row['Close']) else None,
                volume=int(row['Volume']) if not pd.isna(row['Volume']) else None,
                data_source=f"yahoo_finance_historical_group_{group_info['group']}"
            )
            all_bronze_rows.append(bronze_row)
    
    print(f"📊 Processed {len(all_bronze_rows)} rows for group {group_info['group']}")
    return all_bronze_rows

In [0]:
# =============================================================================
# CELL #4: GROUP DUPLICATE PREVENTION (UPDATED)
# PURPOSE: Check which groups already exist and need processing
# =============================================================================

def check_existing_groups():
    """Check which historical groups already exist in Bronze table"""
    
    print("🔍 Checking existing historical groups...")
    
    try:
        # Check for historical group data
        existing_groups_df = spark.sql("""
            SELECT DISTINCT data_source
            FROM bronze_market_data_persistent 
            WHERE data_source LIKE 'yahoo_finance_historical_group_%'
            ORDER BY data_source
        """)
        
        existing_groups = [row['data_source'] for row in existing_groups_df.collect()]
        existing_group_numbers = []
        
        for group_source in existing_groups:
            # Extract group number from data_source
            group_num = int(group_source.split('_')[-1])
            existing_group_numbers.append(group_num)
        
        print(f"📊 Existing groups found: {sorted(existing_group_numbers)}")
        return existing_group_numbers
        
    except Exception as e:
        print(f"⚠️ No existing groups found: {e}")
        return []

def get_groups_to_process(existing_group_numbers):
    """Determine which groups need to be processed"""
    
    all_group_numbers = [group['group'] for group in historical_groups]
    groups_to_process = [num for num in all_group_numbers if num not in existing_group_numbers]
    
    print(f"📋 Groups to process: {sorted(groups_to_process)}")
    print(f"⏭️ Groups already completed: {sorted(existing_group_numbers)}")
    
    return groups_to_process

# Check current status
existing_groups = check_existing_groups()
needed_groups = get_groups_to_process(existing_groups)

print(f"\n🎯 Status: {len(needed_groups)} groups remaining out of {len(historical_groups)} total")


In [0]:
# =============================================================================
# CELL #5: PROCESS GROUP 1 (AMD)
# =============================================================================

GROUP_TO_PROCESS = 1
target_group = historical_groups[0]  # AMD group

print(f"🎯 Processing Group {GROUP_TO_PROCESS}: {target_group['name']}")
print(f"📊 Stocks: {target_group['stocks']}")

if GROUP_TO_PROCESS not in existing_groups:
    group_data = download_group_data(
        target_group['stocks'], 
        target_group['start'], 
        target_group['end'], 
        target_group
    )
    
    if group_data is not None:
        bronze_rows = process_group_to_bronze(group_data, target_group['stocks'], target_group)
        
        if bronze_rows:
            group_df = spark.createDataFrame(bronze_rows, bronze_schema)
            group_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
            
            print(f"✅ Group {GROUP_TO_PROCESS} completed: {group_df.count()} rows saved")
            
            total_count = spark.sql("SELECT COUNT(*) as count FROM bronze_market_data_persistent").collect()[0]['count']
            print(f"📊 Total Bronze records: {total_count}")
else:
    print(f"⚠️ Group {GROUP_TO_PROCESS} already exists")

In [0]:
# =============================================================================
# CELL #6: PROCESS GROUP 2 - 1980s TECH GIANTS (SIMPLE VERSION)
# PURPOSE: Download and process AAPL, ORCL, MSFT (1980-2025)
# =============================================================================

GROUP_TO_PROCESS = 2
target_group = historical_groups[1]  # 1980s Tech Giants

print(f"🎯 Processing Group {GROUP_TO_PROCESS}: {target_group['name']}")
print(f"📊 Stocks: {target_group['stocks']}")

if GROUP_TO_PROCESS not in existing_groups:
    group_data = download_group_data(
        target_group['stocks'], 
        target_group['start'], 
        target_group['end'], 
        target_group
    )
    
    if group_data is not None:
        bronze_rows = process_group_to_bronze(group_data, target_group['stocks'], target_group)
        
        if bronze_rows:
            group_df = spark.createDataFrame(bronze_rows, bronze_schema)
            group_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
            
            print(f"✅ Group {GROUP_TO_PROCESS} completed: {group_df.count()} rows saved")
            
            total_count = spark.sql("SELECT COUNT(*) as count FROM bronze_market_data_persistent").collect()[0]['count']
            print(f"📊 Total Bronze records: {total_count}")
else:
    print(f"⚠️ Group {GROUP_TO_PROCESS} already exists")

In [0]:
# =============================================================================
# CELL #7: PROCESS GROUP 3 - INTERNET & GPU ERA (SIMPLE VERSION)
# PURPOSE: Download and process AMZN, NVDA, CRM, GOOGL (1997-2025)
# =============================================================================

GROUP_TO_PROCESS = 3
target_group = historical_groups[2]  # Internet & GPU Era

print(f"🎯 Processing Group {GROUP_TO_PROCESS}: {target_group['name']}")
print(f"📊 Stocks: {target_group['stocks']}")

if GROUP_TO_PROCESS not in existing_groups:
    group_data = download_group_data(
        target_group['stocks'], 
        target_group['start'], 
        target_group['end'], 
        target_group
    )
    
    if group_data is not None:
        bronze_rows = process_group_to_bronze(group_data, target_group['stocks'], target_group)
        
        if bronze_rows:
            group_df = spark.createDataFrame(bronze_rows, bronze_schema)
            group_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
            
            print(f"✅ Group {GROUP_TO_PROCESS} completed: {group_df.count()} rows saved")
            
            total_count = spark.sql("SELECT COUNT(*) as count FROM bronze_market_data_persistent").collect()[0]['count']
            print(f"📊 Total Bronze records: {total_count}")
else:
    print(f"⚠️ Group {GROUP_TO_PROCESS} already exists")

In [0]:
# =============================================================================
# CELL #8: PROCESS GROUP 4 - SOCIAL & CLOUD ERA (SIMPLE VERSION)
# PURPOSE: Download and process SMCI, META, BOTZ, MDB (2007-2025)
# =============================================================================

GROUP_TO_PROCESS = 4
target_group = historical_groups[3]  # Social & Cloud Era

print(f"🎯 Processing Group {GROUP_TO_PROCESS}: {target_group['name']}")
print(f"📊 Stocks: {target_group['stocks']}")

if GROUP_TO_PROCESS not in existing_groups:
    group_data = download_group_data(
        target_group['stocks'], 
        target_group['start'], 
        target_group['end'], 
        target_group
    )
    
    if group_data is not None:
        bronze_rows = process_group_to_bronze(group_data, target_group['stocks'], target_group)
        
        if bronze_rows:
            group_df = spark.createDataFrame(bronze_rows, bronze_schema)
            group_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
            
            print(f"✅ Group {GROUP_TO_PROCESS} completed: {group_df.count()} rows saved")
            
            total_count = spark.sql("SELECT COUNT(*) as count FROM bronze_market_data_persistent").collect()[0]['count']
            print(f"📊 Total Bronze records: {total_count}")
else:
    print(f"⚠️ Group {GROUP_TO_PROCESS} already exists")

In [0]:
# =============================================================================
# CELL #9: PROCESS GROUP 5 - AI REVOLUTION (SIMPLE VERSION)
# PURPOSE: Download and process SNOW, PLTR, AI (2020-2025)
# =============================================================================

GROUP_TO_PROCESS = 5
target_group = historical_groups[4]  # AI Revolution

print(f"🎯 Processing Group {GROUP_TO_PROCESS}: {target_group['name']}")
print(f"📊 Stocks: {target_group['stocks']}")

if GROUP_TO_PROCESS not in existing_groups:
    group_data = download_group_data(
        target_group['stocks'], 
        target_group['start'], 
        target_group['end'], 
        target_group
    )
    
    if group_data is not None:
        bronze_rows = process_group_to_bronze(group_data, target_group['stocks'], target_group)
        
        if bronze_rows:
            group_df = spark.createDataFrame(bronze_rows, bronze_schema)
            group_df.write.mode("append").saveAsTable("bronze_market_data_persistent")
            
            print(f"✅ Group {GROUP_TO_PROCESS} completed: {group_df.count()} rows saved")
            
            total_count = spark.sql("SELECT COUNT(*) as count FROM bronze_market_data_persistent").collect()[0]['count']
            print(f"📊 Total Bronze records: {total_count}")
            
            # Final celebration summary
            print(f"\n🎉 HISTORICAL DATA COLLECTION COMPLETE! 🎉")
            print(f"📈 Final Bronze table summary:")
            spark.sql("""
                SELECT data_source, COUNT(*) as count
                FROM bronze_market_data_persistent 
                GROUP BY data_source
                ORDER BY data_source
            """).show()
            
else:
    print(f"⚠️ Group {GROUP_TO_PROCESS} already exists")